OR calculation across the CVFG dataframe

In [ ]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

In [ ]:
# Common imports
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

Note that we need to use the bucket path for the WGS table in dataproc, but we point to the local mount in the standard environment. Unsure why the former does't work in both (as of 2026-09-25).

In [ ]:
# Constants (mostly paths to things)

bucket_mount_point = Path.home() / 'workspace/scalable-vus-validation'

variant_table_path = bucket_mount_point / 'variant_tables/annotated_filtered_VEP_calibrations.tsv.gz'
google_project = os.getenv('GOOGLE_PROJECT')
vep_carriers_path = bucket_mount_point / 'carrier_tables/vep-carriers.parquet'
demographics_path = bucket_mount_point / 'cohorts/demo.tsv.gz'
ancestry_pca_path = bucket_mount_point / 'cohorts/ancestry_pca.tsv.gz'
or_estimates_path = bucket_mount_point / 'results/VEP-OR-estimates.parquet'

# In standard env
#wgs_mt_path = f"/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

# In dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'


In [ ]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [ ]:
variants_df

In [ ]:
variants_df['POS'] = pd.to_numeric(variants_df['POS'], downcast='integer')

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [ ]:
variants_df[['consequence']].value_counts(dropna=False)

In [ ]:
score_cols = variants_df.columns[variants_df.columns.str.endswith('score')]
score_cols

In [ ]:
variants_df[score_cols] = variants_df[score_cols].apply(pd.to_numeric)

In [ ]:
points_cols = variants_df.columns[variants_df.columns.str.startswith('points')]
points_cols

In [ ]:
variants_df[points_cols] = variants_df[points_cols].apply(pd.to_numeric, downcast='integer')

In [ ]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path('gene_cohorts', f'{gene} cohort.parquet').exists()
}

For each predictor, we examine the three calibration strategies for both missense only and all variants.

In [ ]:
from cvfgaou.notation import LEQ_CHAR, GEQ_CHAR

predictors = ['AlphaMissense', 'MutPred2', 'REVEL']
calibrators = ['single_gene', 'domain_aggregate']

threshold_lists = { # Bergquist et al. thresholds 10.1016/j.gim.2025.101402
    'MutPred2': [
        (f'{LEQ_CHAR} -4', pd.Series.le, 0.010), # Benign Strong
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.031), # Benign Moderate+
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.197), # Benign Moderate
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.391), # Benign Supporting
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.737), # Pathogenic Supporting
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.829), # Pathogenic Moderate
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.895), # Pathogenic Moderate+
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.932)  # Pathogenic Strong
    ],
    'AlphaMissense': [
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.070), # Benign Moderate+
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.099), # Benign Moderate
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.169), # Benign Supporting
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.792), # Pathogenic Supporting
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.906), # Pathogenic Moderate
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.972), # Pathogenic Moderate+
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.990)  # Pathogenic Strong
    ],
    'REVEL': [
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.932), # Pathogenic strong
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.879), # -- modetate+
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.773), # -- moderate
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.644), # Pathogenic supporting
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.290), # Benign supporting
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.183), # -- moderate
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.052), # -- moderate+
        (f'{LEQ_CHAR} -4', pd.Series.le, 0.016) # -- strong
    ]
}

In [ ]:
# Genome-wide calibration groups
genome_wide_groups = {
    (predictor, gene, 'genome_wide', consequence, classification):
        df[comparator(df[f'{predictor}_score'], threshold) & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for predictor, threshold_list in threshold_lists.items()
    for (classification, comparator, threshold) in threshold_list
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['consequence'].str.contains('missense', case=False, na=False, regex=False))
    )
    if gene in genes_with_cohorts
}

In [ ]:
# Other calibration groups
advanced_calibration_groups = {
    (predictor, gene, calibration, consequence, classification):
        df[classification_selection & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for predictor in predictors
    for calibration in calibrators
    for classification, classification_selection in [
        (f'{GEQ_CHAR} {points:+d}', pd.to_numeric(df[f'points_{predictor}_{calibration}']) >= points)
        for points in range(1,5)
    ] + [
        (f'{LEQ_CHAR} {-points:+d}', pd.to_numeric(df[f'points_{predictor}_{calibration}']) <= -points)
        for points in range(1,9)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['consequence'].str.contains('missense', case=False, na=False, regex=False))
    )
    if gene in genes_with_cohorts
}

In [ ]:
variant_groups = genome_wide_groups | advanced_calibration_groups

Next, get the sets of carriers for each variant group

In [ ]:
for label, df in variant_groups.items():
    if df[['#CHROM', 'POS', 'REF', 'ALT']].isna().any(axis=None):
        display(label)
        display(df)

In [ ]:
import os
import hail as hl

In [ ]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

In [ ]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [ ]:
from cvfgaou import hailtools

In [ ]:
# Attempt to check if we are adding to an existing file
excluded_groups = set()
if vep_carriers_path.exists():
    carriers_df = pd.read_parquet(vep_carriers_path)
    excluded_groups = set(
        carriers_df.
        index.
        to_frame()[
            ['Predictor', 'Gene', 'Calibration', 'Consequence', 'Group']
        ].
        drop_duplicates().
        itertuples(index=False, name=None)
    )
    display(f'Existing file found with {len(excluded_groups)} groups.')
else:
    carriers_df = None

for variant_group, df in tqdm(variant_groups.items()):
    if variant_group in excluded_groups:
        continue
    if df.empty:
        continue
    carrier_df = hailtools.get_cols_with_variants(
        df[['#CHROM', 'POS', 'REF', 'ALT']], wgs_mt,
        contig_col='#CHROM',
        pos_col='POS',
        ref_col='REF',
        alt_col='ALT',
        reference_genome='GRCh38'
    ).set_index('s').assign(**dict(zip(index_cols, variant_group))).set_index(index_cols, append=True)
    
    # `pd.concat` ignores `None`s so this always works
    carriers_df = pd.concat([carriers_df, carrier_df])

    carriers_df.to_parquet(vep_carriers_path)

In [ ]:
#carriers_df = pd.read_parquet(vep_carriers_path)

In [ ]:
carriers_df

OR calculation

In [ ]:
demographics_df = pd.read_table(demographics_path, index_col='person_id')
demographics_df

In [ ]:
pca_df = pd.read_table(ancestry_pca_path, index_col='research_id')
pca_df

In [ ]:
from functools import cache

@cache
def assemble_cohort(gene):
    result = pd.read_parquet(f'gene_cohorts/{gene} cohort.parquet').join(demographics_df).join(pca_df)
    # Trichotomize sex
    result.sex_at_birth.where(
        result.sex_at_birth.isin(['Male', 'Female']),
        'Other',
        inplace=True
    )
    
    return result


In [ ]:
from cvfgaou import odds

In [ ]:
or_estimates_dict = {
    (dataset, gene, grouping, consequence, group):
        odds.estimate_logOR(
            pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer'),
            assemble_cohort(gene)
        )
    for (dataset, gene, grouping, consequence, group), exposure_df
    in tqdm(carriers_df.groupby(['Predictor', 'Gene', 'Calibration', 'Consequence', 'Group']))
}

In [ ]:
or_estimates_df = pd.DataFrame.from_dict(or_estimates_dict, orient='index')
or_estimates_df.index.set_names(['Predictor', 'Gene', 'Calibration', 'Consequence', 'Group'], inplace=True)
or_estimates_df

In [ ]:
or_estimates_df.to_parquet(or_estimates_path)